# Logistic regression: L1, L2 and Elastic Net

Tune shrinkage and class weighting against maintenance cost. Inspect sparsity and coefficient stability rather than treating logistic regression as a token baseline.

In [ ]:
from pathlib import Path
import pandas as pd

from scania_aps.data import TRAIN_FILENAME, TEST_FILENAME, read_raw_csv

ROOT = Path.cwd().resolve()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
TRAIN = ROOT / "data" / "raw" / TRAIN_FILENAME
TEST = ROOT / "data" / "raw" / TEST_FILENAME
ARTIFACTS = ROOT / "artifacts"
assert TRAIN.exists() and TEST.exists(), "Run: poetry run scania-aps download"
train = read_raw_csv(TRAIN)
test = read_raw_csv(TEST)
print(train.X.shape, test.X.shape, train.y.mean(), test.y.mean())

In [ ]:
from scania_aps.optimization import tune_logistic
from scania_aps.split import development_split

split = development_split(train.X, train.y)
best, trace = tune_logistic(split.X_fit, split.y_fit, split.X_tune, split.y_tune, n_trials=36)
print(best)
pd.DataFrame([{
    **r.config.__dict__, "threshold": r.threshold, "tune_cost": r.tune_cost, "pr_auc": r.pr_auc
} for r in trace]).sort_values("tune_cost").head(15)

In [ ]:
from scania_aps.feature_selection import l1_nonzero_features
from scania_aps.models.logistic import build_logistic_pipeline

model = build_logistic_pipeline(best).fit(split.X_fit, split.y_fit)
if best.penalty in {"l1", "elasticnet"}:
    pd.DataFrame([x.__dict__ for x in l1_nonzero_features(model)]).head(30)